# PyTorch - Transformers Experimentation

This notebook is intended to experiment the usage of Transformers in PyTorch for Time Series Forecasting.

# Notebook Setup

## Imports

In [11]:
# Import Standard Libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

## Define Configurations

In [2]:
# Data path
sunspot_data_path = './../../data/raw/sunspot_data.csv'

# Read Data

In [3]:
# Read data from local path
sunspot_data = pd.read_csv(
    sunspot_data_path,
    sep=';',
    header=None,
    names=['year', 'month', 'day', 'dec_year', 'sn_value', 'sn_error', 'obs_num', 'unused1'],
    na_values=['-1'],
    index_col=False
)

# Data Preprocessing

## Sunspot Data

In [4]:
# Find the first id that has a subsequent sequence with at least 1 observation
start_id = max(sunspot_data[sunspot_data['obs_num'] == 0].index.tolist()) + 1

# Split train and test data
sunspot_data_valida = sunspot_data.iloc[start_id:].copy()
sunspot_data['sn_value'] = sunspot_data['sn_value'].astype(float)
sunspot_data_train = sunspot_data[sunspot_data['year'] < 2000]
sunspot_data_test = sunspot_data[sunspot_data['year'] >= 2000]

# Select only the column 'sn_value'
sunspot_data_train = sunspot_data_train['sn_value'].to_numpy().reshape(-1, 1)
sunspot_data_test = sunspot_data_test['sn_value'].to_numpy().reshape(-1, 1)

# Standardisation
scaler = StandardScaler()
sunspot_data_train = scaler.fit_transform(sunspot_data_train).flatten().tolist()
sunspot_data_test = scaler.transform(sunspot_data_test).flatten().tolist()

# Create different sequence batches of 10 time steps each
def to_sequences(sequence_size, observations):
    """Transform a sequence of observations into train and test sequences. (e.g., [1, 2, 3] -> [4])"""
    x, y = [], []
    for i in range(len(observations) - sequence_size):
        # Compute the current window and the subsequent element (i.e., target)
        window = observations[i:(i + sequence_size)]
        after_window = observations[i + sequence_size]

        # Append them
        x.append(window)
        y.append(after_window)

    return (torch.tensor(x, dtype=torch.float32).view(-1, sequence_size, 1),
            torch.tensor(y, dtype=torch.float32).view(-1, 1))

# to_sequence example
example_sequence = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
example_sequence_size = 4
example_sequence_output = to_sequences(example_sequence_size, example_sequence)
print('Example sequence:', example_sequence)
print('Example sequence size:', example_sequence_size)
print('Example sequence output X one element:', example_sequence_output[0][0])
print('Example sequence Output Y one element:', example_sequence_output[1][0])

# Transform train and test into sequences
sunspot_sequence_size = 10
x_train, y_train = to_sequences(sunspot_sequence_size, sunspot_data_train)
x_test, y_test = to_sequences(sunspot_sequence_size, sunspot_data_test)

# Create the Data Loader in PyTorch
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Example sequence: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Example sequence size: 4
Example sequence output X one element: tensor([[1.],
        [2.],
        [3.],
        [4.]])
Example sequence Output Y one element: tensor([5.])


# Model Definition

## Positional Encoding - Sinusoidal Functions

In [16]:
class PositionalEncoder(nn.Module):
    """
    Define a Positional Encoder through sinusoidal functions.

    PE(pos, 2i) = sin(pos/(10000^(2i/d_model)))
    PE(pos, 2i + 1) = cost(pos/(10000^(2i/d_model)))

    pos: position in the sequence
    i: is the dimension index (half of the model dimension d_model)
    d_model: is the model dimension (the embedding dimension)

    NOTE: 2i and 2i + 1 is for separate sine and cosine values into even and odd indicies.
    """
    def __init__(self, embeddings_size, dropout_probability=0.1, max_len_sequence=5000):

        # Initialise the super class
        super(PositionalEncoder, self).__init__()

        # Set the dropout layer
        self.dropout = nn.Dropout(p=dropout_probability)

        # Initialise positional encoding matrix of dimension (max_length_sequence x embeddings_size)
        positional_encoding_matrix = torch.zeros(max_len_sequence, embeddings_size)

        # Create the position from 1 to the max length of the input sequence (reshape x -> (x, 1))
        position = torch.arange(0, max_len_sequence, dtype=torch.float).unsqueeze(1)

        # Create the dividend term as 10000^(2i/d)
        dividend_term = torch.exp(torch.arange(0, embeddings_size, 2).float() * (-np.log(10000.0) / embeddings_size))


In [50]:
# Example of positional encoding
sequence_len = 5
embeddings_size = 3

# Initialise variables
pe = torch.zeros(embeddings_size, sequence_len)
position = torch.arange(0, sequence_len, dtype=torch.float).unsqueeze(1)
dividend_term = torch.arange(0, embeddings_size, 2).float()

# Compute positional encoding for even and odd columns in the Positional Encoding Matrix
#pe[:, 0::2] = torch.sin(position * dividend_term)
#pe[:, 1::2] = torch.cos(position * dividend_term)

print('Sequence Length: ', sequence_len)
print('Embeddings Size: ', embeddings_size)
print('Position: ', position)
print('Dividend term: ', dividend_term)
print('Positional Encoding: ', pe)

Sequence Length:  5
Embeddings Size:  3
Position:  tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.]])
Dividend term:  tensor([0., 2.])
Positional Encoding:  tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


In [45]:
# Column of even indices
pe[:, 0::2] = 3
# Columns of odd indices
pe[:, 1::2] = 2

In [46]:
print(pe)

tensor([[3., 2., 3., 2., 3.],
        [3., 2., 3., 2., 3.],
        [3., 2., 3., 2., 3.]])


# Model Training